In [4]:
import os
import csv
import ast
import re
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, hamming_loss, precision_recall_fscore_support
from sklearn.preprocessing import MultiLabelBinarizer

In [5]:
# Variables globales:
SEED = 2627
DEV_PART = 0.15
CORPUS_DIR = "data"
TRAIN_FILE = os.path.join(CORPUS_DIR, "dataset_train.csv")
TEST_FILE = os.path.join(CORPUS_DIR, "dataset_test.csv")
TOKEN_PATTERN = r"\S+" # Captura cualquier grupo de caracteres que NO contenga espacios

In [6]:
# Función para cargar el corpus:
def load_corpus(filename):
    papers, terms = [], []

    # Abrimos el archivo 'filename' y lo leemos línea por línea:
    with open(filename, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=',')

        # Saltamos la fila con los títulos de las columnas:
        next(reader)

        # Extraemos las columnas del archivo:
        for row in reader:
            if not row or len(row) < 3:
                continue

            papers.append((row[0], row[1]))
            terms.append(ast.literal_eval(row[2]))

    return papers, terms


# Función de tokenización:
def tokenize(text):
    # Buscamos palabras que empiecen por letras y puedan contener guiones
    # (para evitar ruido como "5k" y no excluir estructuras como "state-of-the-art"):
    pattern = re.compile(r"(?u)\b[a-zA-Z][\w-]+\b")
    return " ".join(pattern.findall(text))


# Función para calcular las métricas de evaluación:
def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    hl = hamming_loss(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "hamming_loss": hl,
    }

In [8]:
# Cargamos el corpus, y guardamos las entradas y los términos:
raw_all, y_all = load_corpus(TRAIN_FILE)
print("Papers de entrenamiento:", len(raw_all))

# Creamos una partición aleatoria de entrenamiento/desarrollo:
raw_train, raw_dev, y_train, y_dev = train_test_split(raw_all, y_all, test_size=DEV_PART, shuffle=True, random_state=SEED)

# Tokenizamos los títulos y los resúmenes:
train_tok = [(tokenize(title), tokenize(summary)) for title, summary in raw_train]
dev_tok = [(tokenize(title), tokenize(summary)) for title, summary in raw_dev]

# Convertimos los términos en listas binarias:
mlb = MultiLabelBinarizer()
y_train_bin = mlb.fit_transform(y_train)
y_dev_bin = mlb.transform(y_dev)

Papers de entrenamiento: 43555
